<a href="https://colab.research.google.com/github/mafaiziyas/sp500-market-segmentation-fintech-analytics/blob/main/notebooks/Task_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Task 1.1

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import sys
import json
import datetime
import urllib.request
from io import StringIO
import pandas as pd
import yfinance as yf

In [3]:
# helper func to print section headings
def heading(text):
    print("\n" + "=" * 60)
    print(text)
    print("=" * 60)

# settings
folder = "/content/drive/MyDrive/sp500 CW/data_snapshots"
os.makedirs(folder, exist_ok=True)
start_date = "2021-01-01"
end_date = "2025-01-01"
retrieval_date = datetime.date.today().isoformat()

In [4]:
# get S&P 500 tickers from Wikipedia

heading("2. Ticker List from Wikepedia")
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
html = urllib.request.urlopen(req).read().decode("utf-8")

tables = pd.read_html(StringIO(html))
wiki = tables[0]
sp500 = wiki[["Symbol", "Security", "GICS Sector", "GICS Sub-Industry"]].copy()

# Yahoo uses - instead of . (BRK.B becomes BRK-B)
sp500["YF_Ticker"] = sp500["Symbol"].str.replace(".", "-", regex=False)
sp500.to_csv(f"{folder}/sp500_tickers_{retrieval_date}.csv", index=False)

print(f"Tickers listed : {len(sp500)}")
print(f"Unique tickers : {sp500['YF_Ticker'].nunique()}")
print(f"Saved to       : {folder}/sp500_tickers_{retrieval_date}.csv")

print("\nFirst 5 rows:")
print(sp500.head())


2. Ticker List from Wikepedia
Tickers listed : 503
Unique tickers : 503
Saved to       : /content/drive/MyDrive/sp500 CW/data_snapshots/sp500_tickers_2026-11-23.csv

First 5 rows:
  Symbol             Security             GICS Sector  \
0    MMM                   3M             Industrials   
1    AOS          A. O. Smith             Industrials   
2    ABT  Abbott Laboratories             Health Care   
3   ABBV               AbbVie             Health Care   
4    ACN            Accenture  Information Technology   

                GICS Sub-Industry YF_Ticker  
0        Industrial Conglomerates       MMM  
1               Building Products       AOS  
2           Health Care Equipment       ABT  
3                   Biotechnology      ABBV  
4  IT Consulting & Other Services       ACN  


In [5]:
# benchmark data
heading("3. BENCHMARK (^GSPC)")
bench = yf.download("^GSPC", start=start_date, end=end_date,
                    auto_adjust=False, progress=False)
if isinstance(bench.columns, pd.MultiIndex):
    bench.columns = bench.columns.get_level_values(0)
bench.to_csv(f"{folder}/benchmark_gspc.csv")

print(f"Trading days   : {len(bench)}")
print(f"First date     : {bench.index.min().date()}")
print(f"Last date      : {bench.index.max().date()}")

print("\nFirst 5 rows:")
print(bench.head())


3. BENCHMARK (^GSPC)
Trading days   : 1005
First date     : 2021-01-04
Last date      : 2024-12-31

First 5 rows:
Price         Adj Close        Close         High          Low         Open  \
Date                                                                          
2021-01-04  3700.649902  3700.649902  3769.989990  3662.709961  3764.610107   
2021-01-05  3726.860107  3726.860107  3737.830078  3695.070068  3698.020020   
2021-01-06  3748.139893  3748.139893  3783.040039  3705.340088  3712.199951   
2021-01-07  3803.790039  3803.790039  3811.550049  3764.709961  3764.709961   
2021-01-08  3824.679932  3824.679932  3826.689941  3783.600098  3815.050049   

Price           Volume  
Date                    
2021-01-04  5015000000  
2021-01-05  4591020000  
2021-01-06  6064110000  
2021-01-07  5099160000  
2021-01-08  4773040000  


In [6]:
# share data
heading("4. SHARE DATA DOWNLOAD")
tickers = sp500["YF_Ticker"].tolist()
shares = yf.download(tickers, start=start_date, end=end_date,
                     auto_adjust=False, actions=True,
                     group_by="ticker", progress=False)

first_day = shares.index.min()
last_day = shares.index.max()
print(f"First date     : {first_day.date()}")
print(f"Last date      : {last_day.date()}")
print(f"Trading days   : {len(shares)}")

# period check
assert first_day >= pd.Timestamp(start_date)
assert last_day < pd.Timestamp(end_date)
print("Period check   : passed")

# save one csv per field
fields = ["Open", "High", "Low", "Close", "Adj Close", "Volume",
          "Dividends", "Stock Splits"]
for f in fields:
    table = shares.xs(f, axis=1, level=1)
    table.to_csv(f"{folder}/shares_{f.replace(' ', '_').lower()}.csv")
print(f"Fields saved   : {', '.join(fields)}")

print(shares.shape)
print(shares["AAPL"].head())


4. SHARE DATA DOWNLOAD


ERROR:yfinance:
5 Failed downloads:
ERROR:yfinance:['HONA', 'FDXF', 'Q', 'SNDK', 'VYLR']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


First date     : 2021-01-04
Last date      : 2024-12-31
Trading days   : 1005
Period check   : passed
Fields saved   : Open, High, Low, Close, Adj Close, Volume, Dividends, Stock Splits
(1005, 4014)
Price             Open        High         Low       Close   Adj Close  \
Date                                                                     
2021-01-04  133.520004  133.610001  126.760002  129.410004  125.537102   
2021-01-05  128.889999  131.740005  128.429993  131.009995  127.089256   
2021-01-06  127.720001  131.050003  126.379997  126.599998  122.811195   
2021-01-07  128.360001  131.630005  127.860001  130.919998  127.001923   
2021-01-08  132.429993  132.630005  130.229996  132.050003  128.098129   

Price          Volume  Dividends  Stock Splits  
Date                                            
2021-01-04  143301900        0.0           0.0  
2021-01-05   97664900        0.0           0.0  
2021-01-06  155088000        0.0           0.0  
2021-01-07  109578200        0.0     

In [7]:
# Validation check 1: tickers with no data

heading("5. TICKERS WITH NO PRICE DATA (EXCLUDED)")
no_data = []
for t in tickers:
    if t not in shares.columns.get_level_values(0):
        no_data.append([t, "not returned by Yahoo"])
    elif shares[t]["Close"].notna().sum() == 0:
        no_data.append([t, "no price data in period"])

excluded = pd.DataFrame(no_data, columns=["YF_Ticker", "reason"])

# add company name and the date it joined the index (evidence for the log)
excluded = excluded.merge(
    wiki[["Symbol", "Security", "Date added"]],
    left_on="YF_Ticker", right_on="Symbol", how="left"
).drop(columns="Symbol")
excluded["reason"] = "added to index after study period, no Yahoo price data"
excluded.to_csv(f"{folder}/acquisition_exclusions.csv", index=False)

print(f"Number excluded: {len(excluded)}\n")
print(excluded[["YF_Ticker", "Security", "Date added"]].to_string(index=False))



5. TICKERS WITH NO PRICE DATA (EXCLUDED)
Number excluded: 5

YF_Ticker            Security Date added
     FDXF       FedEx Freight 2026-06-01
     HONA Honeywell Aerospace 2026-06-29
        Q   Qnity Electronics 2025-11-03
     SNDK             Sandisk 2025-11-28
     VYLR               Vylor 2026-10-01


In [8]:
# Validation check 2: partial histories

heading("6. SHARES WITH PARTIAL HISTORY (KEPT FOR NOW)")
close = shares.xs("Close", axis=1, level=1)
counts = close.notna().sum()
short = counts[(counts > 0) & (counts < len(shares))].sort_values()

partial = short.reset_index()
partial.columns = ["YF_Ticker", "trading_days"]
partial["missing_days"] = len(shares) - partial["trading_days"]
partial.to_csv(f"{folder}/partial_history.csv", index=False)

print(f"Full history = {len(shares)} trading days")
print(f"Shares with partial history: {len(partial)}\n")
print(partial.to_string(index=False))



6. SHARES WITH PARTIAL HISTORY (KEPT FOR NOW)
Full history = 1005 trading days
Shares with partial history: 10

YF_Ticker  trading_days  missing_days
      GEV           193           812
     SOLV           194           811
     RDDT           197           808
     VLTO           313           692
     GEHC           513           492
      CEG           742           263
     HOOD           862           143
      APP           935            70
     COIN           936            69
      EXE           979            26


In [9]:
#The exclusion log

heading("5. TICKERS WITH NO PRICE DATA (EXCLUDED)")

no_data = []
for t in tickers:
    if t not in shares.columns.get_level_values(0):
        no_data.append([t, "not returned by Yahoo"])
    elif shares[t]["Close"].notna().sum() == 0:
        no_data.append([t, "no price data in period"])

excluded = pd.DataFrame(no_data, columns=["YF_Ticker", "reason"])
excluded = excluded.merge(wiki[["Symbol", "Security", "Date added"]],
                          left_on="YF_Ticker", right_on="Symbol", how="left").drop(columns="Symbol")
excluded["reason"] = "added to index after study period, no Yahoo price data"
excluded.to_csv(f"{folder}/acquisition_exclusions.csv", index=False)
print(excluded[["YF_Ticker", "Security", "Date added"]].to_string(index=False))


5. TICKERS WITH NO PRICE DATA (EXCLUDED)
YF_Ticker            Security Date added
     FDXF       FedEx Freight 2026-06-01
     HONA Honeywell Aerospace 2026-06-29
        Q   Qnity Electronics 2025-11-03
     SNDK             Sandisk 2025-11-28
     VYLR               Vylor 2026-10-01


In [10]:
heading("6. PROVENANCE RECORD")

info = {
    "source_tickers": url,
    "source_prices": "Yahoo Finance via yfinance",
    "retrieval_date": retrieval_date,
    "start": start_date,
    "end_exclusive": end_date,
    "benchmark": "^GSPC",
    "auto_adjust": False,
    "first_date": str(first_day.date()),
    "last_date": str(last_day.date()),
    "trading_days": len(shares),
    "tickers_listed": len(sp500),
    "tickers_no_data": len(excluded),
    "python_version": sys.version.split()[0],
    "yfinance_version": yf.__version__,
    "pandas_version": pd.__version__,
}

with open(f"{folder}/info.json", "w") as file:
    json.dump(info, file, indent=2)

print("Saved:", f"{folder}/info.json")
print(json.dumps(info, indent=2))


6. PROVENANCE RECORD
Saved: /content/drive/MyDrive/sp500 CW/data_snapshots/info.json
{
  "source_tickers": "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
  "source_prices": "Yahoo Finance via yfinance",
  "retrieval_date": "2026-11-23",
  "start": "2021-01-01",
  "end_exclusive": "2025-01-01",
  "benchmark": "^GSPC",
  "auto_adjust": false,
  "first_date": "2021-01-04",
  "last_date": "2024-12-31",
  "trading_days": 1005,
  "tickers_listed": 503,
  "tickers_no_data": 5,
  "python_version": "3.13.18",
  "yfinance_version": "0.2.66",
  "pandas_version": "2.2.3"
}


#Task 1.2

In [14]:
heading("1.2 DATA VALIDATION & ALLIGMENT")

#reading out the tables as rows = dates, columns = companies
close  = shares.xs("Close", axis=1, level=1)
adj    = shares.xs("Adj Close", axis=1, level=1)
volume = shares.xs("Volume", axis=1, level=1)
splits = shares.xs("Stock Splits", axis=1, level=1)
divs   = shares.xs("Dividends", axis=1, level=1)

#applying the rule: full 1005 days only excluding partial histories
counts = close.notna().sum()
full = counts[counts == len(close)].index.tolist()
print("Shares with full history:", len(full))

close, adj, volume = close[full], adj[full], volume[full]
splits, divs = splits[full], divs[full]

#check 1: duplicate dates
heading("check 1: duplicate dates")
print("Duplicate dates:", close.index.duplicated().sum())

#check 2: missing values in the kept shares
heading("check 2: missing values in the kept shares")
print("Missing Close values:", close.isna().sum().sum())
print("Missing Adj Close values:", adj.isna().sum().sum())
print("Missing Volume values:", volume.isna().sum().sum())

#check 3: prices that are zero or negative
heading("check 3: prices that are zero or negative")
print("Close <= 0:", (close <= 0).sum().sum())
print("Adj Close <= 0:", (adj <= 0).sum().sum())

#check 4: volume errors
heading("check 4: volume errors")
print("Negative volume:", (volume < 0).sum().sum())
zero_vol = (volume == 0).sum()
print("Shares with zero-volume days:", (zero_vol > 0).sum())
print(zero_vol[zero_vol > 0])

#check 5: corporate actions
heading("check 5: corporate actions")
split_events = splits[splits > 0].stack()
print("Split events:", len(split_events))
print(split_events)
print("Dividend events:", (divs > 0).sum().sum())


1.2 DATA VALIDATION & ALLIGMENT
Shares with full history: 488

check 1: duplicate dates
Duplicate dates: 0

check 2: missing values in the kept shares
Missing Close values: 0
Missing Adj Close values: 0
Missing Volume values: 0

check 3: prices that are zero or negative
Close <= 0: 0
Adj Close <= 0: 0

check 4: volume errors
Negative volume: 0
Shares with zero-volume days: 2
Ticker
FERG     27
SW      424
dtype: int64

check 5: corporate actions
Split events: 61
Date        Ticker
2021-03-05  APH        2.000
2021-04-01  SHW        3.000
2021-06-03  MRK        1.048
2021-06-28  CSGP      10.000
2021-06-29  CSX        3.000
                       ...  
2024-10-03  LRCX      10.000
2024-12-04  ANET       4.000
2024-12-13  ETR        2.000
2024-12-16  PANW       2.000
2024-12-20  TSCO       5.000
Length: 61, dtype: float64
Dividend events: 6046


Zero volume for SW (424 days) and FERG (27 days). These could be data quirks, since an S&P 500 share doesn't normally have zero trades on a trading day.

In [12]:
#Investigating check 4: the zero volume days

for t in ["SW", "FERG"]:
    z = volume[t][volume[t] == 0]
    print(t, "| first zero:", z.index.min().date(), "| last zero:", z.index.max().date())
    print("Close on those days:", close[t][z.index].describe()[["min", "max"]].to_dict())

SW | first zero: 2021-01-05 | last zero: 2024-07-05
Close on those days: {'min': 28.040000915527344, 'max': 60.06999969482422}
FERG | first zero: 2021-01-07 | last zero: 2021-03-05
Close on those days: {'min': 118.19999694824219, 'max': 124.19999694824219}


In [17]:
print((volume["SW"] == 0).groupby(volume.index.year).sum())

Date
2021    130
2022     96
2023    121
2024     77
Name: SW, dtype: int64


In [15]:
heading("1.2 INCLUSION AND EXCLUSION LOG")

n_days = len(close)   # shpuld be 1005 trading days in the study period

log = sp500[["Symbol", "YF_Ticker", "Security", "GICS Sector"]].copy()

# number of trading days with a price (0 for the tickers Yahoo had data missing for)
log["trading_days"] = log["YF_Ticker"].map(counts).fillna(0).astype(int)

# date the company joined the index, from the Wikipedia table
log["date_added_to_index"] = log["Symbol"].map(wiki.set_index("Symbol")["Date added"])

# status and reason for each ticker
def decide(days):
    if days == 0:
        return "excluded", "no Yahoo price data in the study period"
    elif days < n_days:
        return "excluded", "partial history: fewer than 1005 trading days"
    else:
        return "included", "full 1005-day history, passed validation checks"

log[["status", "reason"]] = log["trading_days"].apply(lambda d: pd.Series(decide(d)))

# noting the two kept shares with zero-volume days
log["note"] = ""
log.loc[log["YF_Ticker"].isin(["SW", "FERG"]), "note"] = "zero-volume days logged; kept"

# checks that the numbers add up
print(log["status"].value_counts())
print()
print(log.groupby(["status", "reason"]).size())
assert len(log) == 503
assert (log["status"] == "included").sum() == len(full)
assert set(log.loc[log["status"] == "included", "YF_Ticker"]) == set(full)

# show the excluded shares
print()
print(log[log["status"] == "excluded"][["YF_Ticker", "Security", "trading_days", "date_added_to_index", "reason"]]
      .sort_values("trading_days").to_string(index=False))

log.to_csv(f"{folder}/inclusion_exclusion_log.csv", index=False)


1.2 INCLUSION AND EXCLUSION LOG
status
included    488
excluded     15
Name: count, dtype: int64

status    reason                                         
excluded  no Yahoo price data in the study period              5
          partial history: fewer than 1005 trading days       10
included  full 1005-day history, passed validation checks    488
dtype: int64

YF_Ticker             Security  trading_days date_added_to_index                                        reason
     HONA  Honeywell Aerospace             0          2026-06-29       no Yahoo price data in the study period
     FDXF        FedEx Freight             0          2026-06-01       no Yahoo price data in the study period
     SNDK              Sandisk             0          2025-11-28       no Yahoo price data in the study period
        Q    Qnity Electronics             0          2025-11-03       no Yahoo price data in the study period
     VYLR                Vylor             0          2026-10-01       no Yahoo

In [16]:
heading("1.2 SPLIT CHECK")
for date, t in [("2021-03-05", "APH"), ("2021-04-01", "SHW")]:
    d = pd.Timestamp(date)
    w = pd.DataFrame({"Close": close[t], "Volume": volume[t]}).loc[d - pd.Timedelta(days=5): d + pd.Timedelta(days=5)]
    print(t)
    print(w, "\n")


1.2 SPLIT CHECK
APH
                Close    Volume
Date                           
2021-03-01  16.245001  10704800
2021-03-02  16.128750  10886400
2021-03-03  15.712500  13084000
2021-03-04  15.010000  19024800
2021-03-05  15.390000  14359200
2021-03-08  15.335000  12974400
2021-03-09  15.550000   9760400
2021-03-10  15.522500  12685200 

SHW
                 Close   Volume
Date                           
2021-03-29  252.103333  1092600
2021-03-30  248.740005  1084500
2021-03-31  246.003326  1321500
2021-04-01  249.750000  1474800
2021-04-05  255.899994  1371100
2021-04-06  258.950012  1908800 



In [20]:
rows = []
for (d, t), ratio in split_events.items():
    i = close.index.get_loc(d)
    if i == 0:
        continue
    price_chg = close[t].iloc[i] / close[t].iloc[i - 1]
    vol_chg = volume[t].iloc[i] / volume[t].iloc[i - 1]
    rows.append([d.date(), t, ratio, round(price_chg, 3), round(vol_chg, 3)])

chk = pd.DataFrame(rows, columns=["date", "ticker", "split_ratio", "close_change", "volume_change"])

# an unadjusted price would change by about 1/ratio; flag big splits where that happens
big = (chk["split_ratio"] > 1.5) | (chk["split_ratio"] < 0.67)
chk["suspicious"] = big & ((chk["close_change"] - 1 / chk["split_ratio"]).abs() < 0.1 * (1 / chk["split_ratio"]).abs())
print("Splits checked:", len(chk), "| suspicious:", int(chk["suspicious"].sum()))
print(chk[chk["suspicious"]])
chk.to_csv(f"{folder}/split_check.csv", index=False)

Splits checked: 61 | suspicious: 0
Empty DataFrame
Columns: [date, ticker, split_ratio, close_change, volume_change, suspicious]
Index: []


In [19]:
heading("1.2 WEEKLY RESAMPLING AND ALIGNMENT")

# benchmark daily adjusted close (read from the saved snapshot)
bench = pd.read_csv(f"{folder}/benchmark_gspc.csv", index_col=0, parse_dates=True)
bench_adj = bench["Adj Close"]

last_day = adj.index.max()

# 1. Friday-ending weeks, using the last available trading day in each week
adj_w = adj.resample("W-FRI").last()
bench_w = bench_adj.resample("W-FRI").last()

# 2. drop the incomplete final week (its Friday label is after our last trading day)
adj_w = adj_w[adj_w.index <= last_day]
bench_w = bench_w[bench_w.index <= last_day]

# check the first week is a full week (expect 5 trading days, 4 to 8 Jan 2021)
first_week_days = ((adj.index >= "2021-01-04") & (adj.index <= adj_w.index[0])).sum()
print("First weekly label:", adj_w.index[0].date(), "| trading days in it:", first_week_days)
print("Last weekly label :", adj_w.index[-1].date())

# 3. weekly returns (no filling)
ret_w = adj_w.pct_change(fill_method=None).iloc[1:]
bench_ret_w = bench_w.pct_change(fill_method=None).iloc[1:]

# 4. alignment checks
print("Weekly prices      :", len(adj_w))
print("Weekly returns     :", len(ret_w))
print("Benchmark returns  :", len(bench_ret_w))
print("Identical week labels:", ret_w.index.equals(bench_ret_w.index))
print("Missing returns    :", int(ret_w.isna().sum().sum()), "| benchmark:", int(bench_ret_w.isna().sum()))
print("Shares in weekly table:", ret_w.shape[1])
print("Weekly return range:", round(ret_w.min().min(), 3), "to", round(ret_w.max().max(), 3))

# 5. save
adj_w.to_csv(f"{folder}/weekly_adj_close.csv")
ret_w.to_csv(f"{folder}/weekly_returns.csv")
bench_ret_w.rename("benchmark_return").to_csv(f"{folder}/benchmark_weekly_returns.csv")
pd.Series(ret_w.columns, name="YF_Ticker").to_csv(f"{folder}/valid_shares_common_set.csv", index=False)


1.2 WEEKLY RESAMPLING AND ALIGNMENT
First weekly label: 2021-01-08 | trading days in it: 5
Last weekly label : 2024-12-27
Weekly prices      : 208
Weekly returns     : 207
Benchmark returns  : 207
Identical week labels: True
Missing returns    : 0 | benchmark: 0
Shares in weekly table: 488
Weekly return range: -0.505 to 0.86


In [21]:
s = ret_w.stack()
print(s.nsmallest(5))
print(s.nlargest(5))

Date        Ticker
2021-03-26  SKYD     -0.504571
2024-04-12  GL       -0.455141
2024-11-01  SMCI     -0.448911
2022-11-04  TWLO     -0.434656
2024-07-26  DXCM     -0.426831
dtype: float64
Date        Ticker
2023-02-03  CVNA      0.859717
2024-11-22  SMCI      0.784177
2022-08-05  CVNA      0.611664
2024-11-15  BE        0.605163
2023-01-13  CVNA      0.591837
dtype: float64


#Task 1.3

In [23]:
import numpy as np

heading("1.3.1 MEAN WEEKLY RETURN")

#The Arithmetic mean of the weekly returns for each share
n_weeks = ret_w.notna().sum()
mean_ret = ret_w.mean()

print("Weekly observations per share:", n_weeks.min(), "to", n_weeks.max())
print(mean_ret.describe().round(5))
print("\nHighest mean weekly return:\n", mean_ret.nlargest(5).round(5))
print("\nLowest mean weekly return:\n", mean_ret.nsmallest(5).round(5))

# check 1: 2 Apr 2021 was Good Friday (market closed),so that week shouldbe use Thursday's price
d = pd.Timestamp("2021-04-02")
print("\nIs 2 Apr 2021 a trading day?", d in adj.index)
print("Weekly price on 2 Apr:", adj_w.loc[d, "AAPL"], "| daily price on 1 Apr:", adj.loc["2021-04-01", "AAPL"])

# check 2: recomputing the means a second way, straight from the weekly price table
alt = (adj_w.values[1:] / adj_w.values[:-1] - 1).mean(axis=0)
print("Max difference, all shares:", np.abs(alt - mean_ret.values).max())

# check 3: one share worked by hand
t = "AAPL"
p = adj_w[t]
print("\nAAPL week 1 return by hand:", p.iloc[1] / p.iloc[0] - 1, "| in table:", ret_w[t].iloc[0])


1.3.1 MEAN WEEKLY RETURN
Weekly observations per share: 207 to 207
count    488.00000
mean       0.00286
std        0.00265
min       -0.00447
25%        0.00118
50%        0.00263
75%        0.00421
max        0.01777
dtype: float64

Highest mean weekly return:
 YF_Ticker
SMCI    0.01777
NVDA    0.01362
VRT     0.01258
VST     0.01139
FIX     0.01130
dtype: float64

Lowest mean weekly return:
 YF_Ticker
EL     -0.00447
BAX    -0.00380
DG     -0.00352
PYPL   -0.00322
ILMN   -0.00315
dtype: float64

Is 2 Apr 2021 a trading day? False
Weekly price on 2 Apr: 119.49723815917969 | daily price on 1 Apr: 119.49723815917969
Max difference, all shares: 0.0

AAPL week 1 return by hand: -0.037183150667912335 | in table: -0.037183150667912335


In [24]:
heading("1.3.2 WEEKLY BETA")

# weeks must match exactly
assert ret_w.index.equals(bench_ret_w.index)
print("Aligned weekly observations:", len(ret_w))

# beta = covariance(share, benchmark) / variance(benchmark)
bench_var = bench_ret_w.var()
cov = ret_w.apply(lambda s: s.cov(bench_ret_w))
beta = cov / bench_var

print("Benchmark weekly variance:", bench_var)
print(beta.describe().round(3))
print("\nHighest beta:\n", beta.nlargest(5).round(3))
print("\nLowest beta:\n", beta.nsmallest(5).round(3))

# check 1: the benchmark's beta against itself must be 1
print("\nBenchmark beta vs itself:", bench_ret_w.cov(bench_ret_w) / bench_var)

# check 2: the regression slope gives the same number
x = bench_ret_w.values
slopes = np.array([np.polyfit(x, ret_w[c].values, 1)[0] for c in ret_w.columns])
print("Max difference vs regression slope:", np.abs(slopes - beta.values).max())

# check 3: price basis of the benchmark
print("Benchmark max |Close - Adj Close|:", (bench["Close"] - bench["Adj Close"]).abs().max())

# save the first two features
features = pd.DataFrame({"mean_weekly_return": mean_ret, "weekly_beta": beta, "weekly_obs": n_weeks})
features.index.name = "YF_Ticker"
features.to_csv(f"{folder}/features_return_beta.csv")
print("\nSaved features_return_beta.csv", features.shape)


1.3.2 WEEKLY BETA
Aligned weekly observations: 207
Benchmark weekly variance: 0.0004921832477954451
count    488.000
mean       0.974
std        0.407
min        0.055
25%        0.667
50%        0.951
75%        1.179
max        3.779
dtype: float64

Highest beta:
 YF_Ticker
CVNA    3.779
XYZ     2.505
COHR    2.139
NVDA    2.126
TWLO    2.073
dtype: float64

Lowest beta:
 YF_Ticker
GIS    0.055
SJM    0.176
MRK    0.198
BMY    0.225
KHC    0.227
dtype: float64

Benchmark beta vs itself: 1.0
Max difference vs regression slope: 2.220446049250313e-15
Benchmark max |Close - Adj Close|: 0.0

Saved features_return_beta.csv (488, 3)


#Task 1.4

In [25]:
heading("1.4 OBSERVATION DATE AND HIGH-VOLUME EVENT FLAGS")

#the observation date = the last common trading day
obs_date = volume.index.max()
print("Observation date:", obs_date.date())

# Average volume of the previous 20 sessions, excluding the day being tested
baseline = volume.shift(1).rolling(20).mean()
threshold = 2 * baseline

# a high-volume day has volume greater than twice the baseline
event = volume > threshold

print("Shares:", event.shape[1], "| days:", event.shape[0])
print("High-volume days in total:", int(event.sum().sum()))
print("Share of days flagged:", round(event.sum().sum() / event.size, 4))



1.4 OBSERVATION DATE AND HIGH-VOLUME EVENT FLAGS
Observation date: 2024-12-31
Shares: 488 | days: 1005
High-volume days in total: 18496
Share of days flagged: 0.0377


#Task 1.4.1

In [26]:
heading("1.4.1 RECENCY")

# latest event date for each share ( is missing if the share has no event)
last_event = {}
for t in event.columns:
    if event[t].any():
        last_event[t] = volume.index[event[t].values].max()
    else:
        last_event[t] = pd.NaT
last_event = pd.to_datetime(pd.Series(last_event))

#Calendar days between the latest event and observation date
recency = (obs_date - last_event).dt.days

print(recency.describe())
print("\nShares with no event (R missing):", int(recency.isna().sum()))
print("Shares with an event on the observation date (R = 0):", int((recency == 0).sum()))
print("Most recent:\n", recency.nsmallest(5))
print("Least recent:\n", recency.nlargest(5))

# Validation checks
# 1. no event can be flagged in the first 20 sessions (not enough history for the baseline)
print("\nEvents in the first 20 sessions:", int(event.iloc[:20].sum().sum()))

#2. cases where the baseline is zero (could create false events)
print("Days with a zero baseline:", int((baseline == 0).sum().sum()))

#3. one share checked by hand
t = "AAPL"
d = last_event[t]
i = volume.index.get_loc(d)
manual_base = volume[t].iloc[i - 20:i].mean() # the 20 sessions before the event day
print("\n", t, "latest event:", d.date())
print("Volume that day:", volume[t].iloc[i])
print("20-session mean before it:", round(manual_base))
print("Threshold (2x):", round(2 * manual_base))
print("Qualifies:", volume[t].iloc[i] > 2 * manual_base)
print("Recency by hand:", (obs_date - d).days, "| in table:", recency[t])
print("Any later event?", bool(event[t].iloc[i + 1:].any()))

#4. Smurfit Westrock (SW) had zero-volume days, so check its events
print("\nSW latest event:", last_event["SW"].date(), "| R =", recency["SW"])

# save to csv
event.astype(int).to_csv(f"{folder}/rfm_event_flags.csv")
recency.rename("recency_days").to_csv(f"{folder}/recency.csv")


1.4.1 RECENCY
count    488.000000
mean      20.389344
std       21.302843
min        4.000000
25%       11.000000
50%       11.000000
75%       15.000000
max      222.000000
dtype: float64

Shares with no event (R missing): 0
Shares with an event on the observation date (R = 0): 0
Most recent:
 NRG      4
AJG      4
SRE      5
LII     11
HBAN    11
dtype: int64
Least recent:
 NVDA    222
WEC     151
MDT     102
DUK     102
HRL     102
dtype: int64

Events in the first 20 sessions: 0
Days with a zero baseline: 0

 AAPL latest event: 2024-12-20
Volume that day: 147495300
20-session mean before it: 45004605
Threshold (2x): 90009210
Qualifies: True
Recency by hand: 11 | in table: 11
Any later event? False

SW latest event: 2024-12-20 | R = 11


In [27]:
print("Shares with R = 11:", int((recency == 11).sum()), "of", len(recency))
print("Events on 20 Dec 2024:", int(event.loc["2024-12-20"].sum()), "of", event.shape[1])

Shares with R = 11: 354 of 488
Events on 20 Dec 2024: 355 of 488


#Task 1.4.2

In [29]:
# 1.4.2 Frequency: high-volume days in the latest 60 days
window = 60
win_days = event.index[-window:]

frequency = event.loc[win_days].sum()

print("Window start:", win_days[0].date())
print("Window end  :", win_days[-1].date())
print("Sessions in window:", len(win_days))

#Check 1: every window day has a valid 20-session baseline
print("Missing baselines in window:", int(baseline.loc[win_days].isna().sum().sum()))
print("Sessions available before window:", len(event) - window)

# Check 2: boundary -the day just before the window is not counted
before = event.index[-window - 1]
print("Day before window:", before.date(), "| events that day (not counted):", int(event.loc[before].sum()))

# Check 3: worked example by hand (AAPL)
t = "AAPL"
vol_w = volume.loc[win_days, t]
thr_w = threshold.loc[win_days, t]
hits = vol_w[vol_w > thr_w]
print("\nAAPL high-volume days in window:")
print(pd.DataFrame({"volume": hits, "threshold": thr_w[hits.index]}))
print("By hand:", len(hits), "| table:", int(frequency[t]))

# Check 4: consistency with Recency
in_window = last_event >= win_days[0]
print("\nMismatches between R and F:", int((in_window != (frequency > 0)).sum()))

# Summary
print("\nFrequency summary:")
print(frequency.describe())
print("Shares with F = 0:", int((frequency == 0).sum()))
print("Most common F values:")
print(frequency.value_counts().head(5))

frequency.rename("frequency").to_csv(f"{folder}/frequency.csv")

Window start: 2024-10-07
Window end  : 2024-12-31
Sessions in window: 60
Missing baselines in window: 0
Sessions available before window: 945
Day before window: 2024-10-04 | events that day (not counted): 2

AAPL high-volume days in window:
               volume   threshold
Date                             
2024-11-25   90152800  87641480.0
2024-12-20  147495300  90009210.0
By hand: 2 | table: 2

Mismatches between R and F: 0

Frequency summary:
count    488.000000
mean       2.987705
std        1.328820
min        0.000000
25%        2.000000
50%        3.000000
75%        4.000000
max        8.000000
dtype: float64
Shares with F = 0: 10
Most common F values:
3    150
2    131
4     94
5     45
1     42
Name: count, dtype: int64


#Task 1.4.3

In [30]:
# 1.4.3 Monetary: approx. traded value (Close x Volume) over the latest 60 sessions
traded_value = close.loc[win_days] * volume.loc[win_days]#per share, per day (USD)
monetary = traded_value.sum() # one number per share

# Check 1: missing values, zero volume, non-positive M
print("Missing Close or Volume in window:", int(close.loc[win_days].isna().sum().sum() + volume.loc[win_days].isna().sum().sum()))
print("Zero-volume days in window:", int((volume.loc[win_days] == 0).sum().sum()))
print("Shares with M <= 0:", int((monetary <= 0).sum()))

# Check 2: split events inside the window (Close and Volume are both split-adjusted by Yahoo)
split_in_window = splits.loc[win_days]
print("Split events in window:", int((split_in_window != 0).sum().sum()))

#Check 3: worked example by hand (AAPL, first 3 days of the window)
t = "AAPL"
print("\nAAPL first 3 window days:")
print(pd.DataFrame({"close": close.loc[win_days, t].head(3),
                    "volume": volume.loc[win_days, t].head(3),
                    "value": traded_value[t].head(3)}))
print("AAPL M (USD bn):", round(monetary[t] / 1e9, 2))

# Check 4: second method, plain numpy
m_np = (close.loc[win_days].values * volume.loc[win_days].values).sum(axis=0)
print("Max difference vs numpy:", float(abs(m_np - monetary.values).max()))

#Summary (USD billions)
print("\nMonetary summary (USD bn):")
print((monetary / 1e9).describe())
print("\nTop 5:");    print((monetary.sort_values(ascending=False).head(5) / 1e9).round(1))
print("\nBottom 5:"); print((monetary.sort_values().head(5) / 1e9).round(2))
print("Max / min ratio:", round(monetary.max() / monetary.min()))
print("Skewness:", round(monetary.skew(), 2))

monetary.rename("monetary_usd").to_csv(f"{folder}/monetary.csv")

Missing Close or Volume in window: 0
Zero-volume days in window: 0
Shares with M <= 0: 0
Split events in window: 4

AAPL first 3 window days:
                 close    volume         value
Date                                          
2024-10-07  221.690002  39505400  8.757952e+09
2024-10-08  225.770004  31855700  7.192062e+09
2024-10-09  229.539993  33591100  7.710501e+09
AAPL M (USD bn): 623.3
Max difference vs numpy: 0.0

Monetary summary (USD bn):
count     488.000000
mean       39.302897
std       128.883283
min         1.297483
25%        10.231942
50%        16.683822
75%        32.100480
max      1847.697379
dtype: float64

Top 5:
Ticker
TSLA    1847.7
NVDA    1825.3
AAPL     623.3
MSFT     534.9
AMZN     468.2
dtype: float64

Bottom 5:
Ticker
NWS     1.30
ECHO    2.35
FOX     2.71
IVZ     3.69
L       3.71
dtype: float64
Max / min ratio: 1424
Skewness: 11.71


#Task 1.5

In [31]:
# 1.5 Combine all features: one row per included share
feature_table = features[["mean_weekly_return", "weekly_beta"]].copy()
feature_table["recency_days"] = recency
feature_table["frequency"]    = frequency
feature_table["monetary_usd"] = monetary
feature_table.index.name = "Ticker"

#Checks
print("Shape:", feature_table.shape)
print("Missing values per column:")
print(feature_table.isna().sum())
print("Duplicate tickers:", int(feature_table.index.duplicated().sum()))
print("Same shares as common valid set:", set(feature_table.index) == set(full))

#Same window and date used for R, F and M
print("\nObservation date:", obs_date.date())
print("Window:", win_days[0].date(), "to", win_days[-1].date(), "| sessions:", len(win_days))

feature_table.to_csv(f"{folder}/feature_table.csv")

Shape: (488, 5)
Missing values per column:
mean_weekly_return    0
weekly_beta           0
recency_days          0
frequency             0
monetary_usd          0
dtype: int64
Duplicate tickers: 0
Same shares as common valid set: True

Observation date: 2024-12-31
Window: 2024-10-07 to 2024-12-31 | sessions: 60


In [32]:
#Evidence tables print
# Feature summaries (one table for the report)
summary = feature_table.describe().T[["count", "mean", "50%", "min", "max"]]
summary = summary.rename(columns={"50%": "median"})
summary["skewness"] = feature_table.skew().round(2)
print(summary.round(4))

# Worked example: one share
t = "AAPL"
print("\nWorked example:", t)
print(feature_table.loc[t])
print("Last event:", last_event[t].date(), "-> R =", int(recency[t]))
print("Events in window:", [d.date() for d in event.loc[win_days, t][event.loc[win_days, t]].index], "-> F =", int(frequency[t]))
print("M (USD bn):", round(monetary[t] / 1e9, 1))

# Inclusion / exclusion log totals
inc = pd.read_csv(f"{folder}/inclusion_exclusion_log.csv")
print("\nLog status counts:")
print(inc["status"].value_counts())

                    count          mean        median           min  \
mean_weekly_return  488.0  2.900000e-03  2.600000e-03 -4.500000e-03   
weekly_beta         488.0  9.739000e-01  9.506000e-01  5.550000e-02   
recency_days        488.0  2.038930e+01  1.100000e+01  4.000000e+00   
frequency           488.0  2.987700e+00  3.000000e+00  0.000000e+00   
monetary_usd        488.0  3.930290e+10  1.668382e+10  1.297483e+09   

                             max  skewness  
mean_weekly_return  1.780000e-02      0.87  
weekly_beta         3.778600e+00      1.10  
recency_days        2.220000e+02      3.65  
frequency           8.000000e+00      0.43  
monetary_usd        1.847697e+12     11.71  

Worked example: AAPL
mean_weekly_return    3.929799e-03
weekly_beta           1.174836e+00
recency_days          1.100000e+01
frequency             2.000000e+00
monetary_usd          6.233034e+11
Name: AAPL, dtype: float64
Last event: 2024-12-20 -> R = 11
Events in window: [datetime.date(2024, 11, 25)

In [33]:
#Table 1: AAPL row from feature_table.csv
saved = pd.read_csv(f"{folder}/feature_table.csv", index_col="Ticker")
saved.loc[["AAPL"]]

,mean_weekly_return,weekly_beta,recency_days,frequency,monetary_usd
Ticker,,,,,
AAPL,0.00393,1.174836,11,2,6.233034e+11


In [35]:
#Table 2: feature summaries, read from feature_table.csv created above.
saved = pd.read_csv(f"{folder}/feature_table.csv", index_col="Ticker")
saved["monetary_usd_bn"] = saved.pop("monetary_usd") / 1e9   # USD billions

table2 = saved.describe().T[["mean", "50%", "min", "max"]].rename(columns={"50%": "median"})
table2["skew"] = saved.skew()
table2["missing"] = saved.isna().sum()

table2.index = ["Mean weekly return", "Weekly beta", "Recency (days)",
                "Frequency (days)", "Monetary (USD bn)"]
table2.round(4)

,mean,median,min,max,skew,missing
Mean weekly return,0.0029,0.0026,-0.0045,0.0178,0.8692,0
Weekly beta,0.9739,0.9506,0.0555,3.7786,1.1043,0
Recency (days),20.3893,11.0000,4.0000,222.0000,3.6534,0
Frequency (days),2.9877,3.0000,0.0000,8.0000,0.4286,0
Monetary (USD bn),39.3029,16.6838,1.2975,1847.6974,11.7073,0


In [36]:
inc = pd.read_csv(f"{folder}/inclusion_exclusion_log.csv")
print(inc.columns.tolist())

# Totals (shows 503 = 488 + 15)
print(inc["status"].value_counts())

# First rows of the log
inc.head()

['Symbol', 'YF_Ticker', 'Security', 'GICS Sector', 'trading_days', 'date_added_to_index', 'status', 'reason', 'note']
status
included    488
excluded     15
Name: count, dtype: int64


,Symbol,YF_Ticker,Security,GICS Sector,trading_days,date_added_to_index,status,reason,note
0,MMM,MMM,3M,Industrials,1005,1957-03-04,included,"full 1005-day history, passed validation checks",NaN
1,AOS,AOS,A. O. Smith,Industrials,1005,2017-07-26,included,"full 1005-day history, passed validation checks",NaN
2,ABT,ABT,Abbott Laboratories,Health Care,1005,1957-03-04,included,"full 1005-day history, passed validation checks",NaN
3,ABBV,ABBV,AbbVie,Health Care,1005,2012-12-31,included,"full 1005-day history, passed validation checks",NaN
4,ACN,ACN,Accenture,Information Technology,1005,2011-07-06,included,"full 1005-day history, passed validation checks",NaN


In [37]:
# The 15 excluded shares
inc[inc["status"] == "excluded"]

,Symbol,YF_Ticker,Security,GICS Sector,trading_days,date_added_to_index,status,reason,note
40,APP,APP,AppLovin,Communication Services,935,2025-09-22,excluded,partial history: fewer than 1005 trading days,NaN
116,COIN,COIN,Coinbase,Financials,936,2025-05-19,excluded,partial history: fewer than 1005 trading days,NaN
123,CEG,CEG,Constellation Energy,Utilities,742,2022-02-02,excluded,partial history: fewer than 1005 trading days,NaN
181,EXE,EXE,Expand Energy,Energy,979,2025-03-24,excluded,partial history: fewer than 1005 trading days,NaN
192,FDXF,FDXF,FedEx Freight,Industrials,0,2026-06-01,excluded,no Yahoo price data in the study period,NaN
210,GEHC,GEHC,GE HealthCare,Health Care,513,2023-01-04,excluded,partial history: fewer than 1005 trading days,NaN
211,GEV,GEV,GE Vernova,Industrials,193,2024-04-02,excluded,partial history: fewer than 1005 trading days,NaN
233,HONA,HONA,Honeywell Aerospace,Industrials,0,2026-06-29,excluded,no Yahoo price data in the study period,NaN
379,Q,Q,Qnity Electronics,Information Technology,0,2025-11-03,excluded,no Yahoo price data in the study period,NaN
385,RDDT,RDDT,Reddit,Communication Services,197,2026-08-18,excluded,partial history: fewer than 1005 trading days,NaN


In [38]:
# The two kept-but-flagged shares
inc[inc["YF_Ticker"].isin(["SW", "FERG"])]

,Symbol,YF_Ticker,Security,GICS Sector,trading_days,date_added_to_index,status,reason,note
193,FERG,FERG,Ferguson Enterprises,Industrials,1005,2026-08-05,included,"full 1005-day history, passed validation checks",zero-volume days logged; kept
413,SW,SW,Smurfit Westrock,Materials,1005,2024-07-08,included,"full 1005-day history, passed validation checks",zero-volume days logged; kept
